[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iamjamesbowden/AG952/blob/main/assignments/march2026/AG952_Assignment_2026.ipynb)

# AG952 | Coursework Assignment 2026

**Textual Analytics for Accounting and Finance**  
University of Strathclyde Business School

---

This notebook serves three purposes: it allocates you to one of four research scenarios based on your student number; it provides the data loading and analytical infrastructure required to complete the analysis; and it records your methodological choices to a central log so that the module team can monitor approach diversity across the cohort. All analytical decisions and their justifications are your own. Your written report must explain and defend every choice made within this notebook, with explicit reference to the methodological literature.

Before you begin, save a personal copy of this notebook to your Google Drive (File > Save a copy in Drive) or to a personal GitHub repository. Do not modify the shared repository copy. Work only from your personal copy throughout the assignment.

**Estimated total time:** 8--12 hours across multiple sessions.  
**Release date:** 10 March 2026  
**Submission deadline:** 1 April 2026

In [ ]:
#@title Step 0 -- Clone the AG952 repository (run this first)

import os
import subprocess

REPO_URL = "https://github.com/iamjamesbowden/AG952.git"
REPO_DIR = "AG952"

if not os.path.exists(REPO_DIR):
    print("Cloning AG952 repository...")
    result = subprocess.run(["git", "clone", REPO_URL], capture_output=True, text=True)
    if result.returncode == 0:
        print("Repository cloned successfully.")
    else:
        print("Clone failed:")
        print(result.stderr)
else:
    print("Repository already present. Pulling latest changes...")
    result = subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", "main"],
                            capture_output=True, text=True)
    print(result.stdout.strip() or "Already up to date.")

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
#@title Configuration -- set APPS_SCRIPT_URL before running

# Paste the deployed Apps Script web app URL here before running the notebook.
# Leave as an empty string if the endpoint has not yet been deployed.
APPS_SCRIPT_URL = "https://script.google.com/macros/s/AKfycbyrv8mSp2e2OpH4nqclZbmTNaRBO_xjMWoc1A8_0Y6bOwk6qfWlXnqH6HCtGpPigXDS/exec"

# Paths -- do not modify
# CORPUS_PATH is set dynamically in the Load Corpus cell after scenario allocation.
CORPUS_BASE_PATH  = "assignments/march2026/data"     # do not modify
LM_DICT_PATH      = "assignments/march2026/data/lm_dictionary.csv"
HARVARD_DICT_PATH = "assignments/march2026/data/harvard_iv_dictionary.csv"
ALLOCATION_PATH   = "assignments/march2026/student_allocations.csv"
SEED_LABELS_PATH  = "assignments/march2026/nb_seed_labels.csv"

In [ ]:
#@title Step 1 -- Install and import dependencies

!pip install requests pandas numpy matplotlib seaborn scikit-learn nltk -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import os
import json
import time
import warnings
import requests
import random

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import nltk
nltk.download("stopwords", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("wordnet", quiet=True)

warnings.filterwarnings("ignore")

print("Dependencies loaded successfully.")

## Scenario Allocation

Each student is assigned to one of four research scenarios based on their student number. The scenario determines which corpus and research question you will work with throughout the assignment. You must not change your allocated scenario.

In [ ]:
#@title Step 2 -- Enter your student number to receive your scenario allocation

SCENARIOS = {
    "A": {
        "id": "A",
        "title": "Climate and ESG Risk Language in the US Energy Sector (2019-2023)",
        "question": (
            "How has the linguistic framing of climate and ESG-related risk evolved "
            "across fossil fuel and renewable energy firms in their 10-K filings between "
            "2019 and 2023, and to what extent does language diverge systematically "
            "between firm types?"
        ),
        "corpus_note": (
            "138 10-K filings from 30 firms across three energy sub-sectors: "
            "12 oil and gas producers, 10 integrated utilities, and 8 pure-play "
            "renewables. Filing years span 2019-2023, though not every firm "
            "has a filing in every year."
        ),
    },
    "B": {
        "id": "B",
        "title": "Narrative Predictors of Corporate Financial Distress (2015-2023)",
        "question": (
            "Do the linguistic features of 10-K filings in the one to three years preceding "
            "a bankruptcy filing differ systematically from those of matched non-distressed "
            "peers, and can those features be used to train a reliable text-based classifier "
            "of financial distress?"
        ),
        "corpus_note": (
            "175 10-K filings from 38 firms: 18 distressed firms (filings from the years "
            "preceding bankruptcy) matched to 20 non-distressed peers by SIC code and "
            "asset quintile. Filing years span 2015-2022."
        ),
    },
    "C": {
        "id": "C",
        "title": "Risk Disclosure and the 2023 US Regional Banking Crisis",
        "question": (
            "Did the risk disclosures in 10-K filings from US regional banks in the period "
            "2020-2022 provide investors with meaningful warning of the vulnerabilities that "
            "materialised during the March 2023 banking crisis, and how does the language "
            "of subsequently failed banks compare to that of survivors?"
        ),
        "corpus_note": (
            "159 10-K filings from 37 US banks across four categories: 3 subsequently "
            "failed banks, 7 stressed survivors, 18 unaffected regional banks, and "
            "9 large systemic institutions. Filing years span 2019-2023, though not "
            "every firm has a filing in every year."
        ),
    },
    "D": {
        "id": "D",
        "title": "Supply Chain Risk Disclosure Before and After COVID-19 (2019-2022)",
        "question": (
            "How did the language of supply chain risk disclosure in 10-K filings change "
            "across manufacturing and logistics-intensive firms in the two years before and "
            "the two years after the onset of the COVID-19 pandemic, and does the degree "
            "of linguistic change vary with firms' ex post supply chain vulnerability?"
        ),
        "corpus_note": (
            "175 10-K filings from 38 firms across five supply-chain-intensive sectors: "
            "6 automotive, 5 consumer electronics, 13 industrial manufacturing, "
            "7 logistics, and 7 retail. Filing years span 2019-2023, though not every "
            "firm has a filing in every year."
        ),
    }
}

# Modulo mapping for fallback allocation
MODULO_MAP = {0: "A", 1: "B", 2: "C", 3: "D"}


STUDENT_ID = input("Enter your student number: ").strip()


def allocate_scenario(student_id, allocation_path, modulo_map):
    """
    Look up student_id in the allocation CSV.
    Falls back to int(student_id) % 4 if the student ID is not found.
    Returns a scenario key: "A", "B", "C", or "D".
    """
    try:
        df = pd.read_csv(allocation_path, dtype=str)
        df["student_id"] = df["student_id"].str.strip()
        match = df[df["student_id"] == student_id.strip()]
        if not match.empty:
            return match.iloc[0]["scenario"].strip().upper(), "allocation file"
    except FileNotFoundError:
        print(f"Warning: allocation file not found at {allocation_path}.")
    except Exception as ex:
        print(f"Warning: could not read allocation file ({ex}).")

    # Fallback: modulo allocation
    try:
        numeric_id = int("".join(filter(str.isdigit, student_id)))
        scenario = modulo_map[numeric_id % 4]
        return scenario, "modulo fallback (student ID not found in allocation file)"
    except Exception:
        return "A", "default fallback (student ID could not be parsed)"


ASSIGNED_SCENARIO_KEY, ALLOCATION_METHOD = allocate_scenario(
    STUDENT_ID, ALLOCATION_PATH, MODULO_MAP
)
SCENARIO = SCENARIOS[ASSIGNED_SCENARIO_KEY]

print("\n" + "=" * 60)
print(f"  Student ID:  {STUDENT_ID}")
print(f"  Scenario:    {SCENARIO['id']} -- {SCENARIO['title']}")
print(f"  Allocated via: {ALLOCATION_METHOD}")
print("=" * 60)
print(f"\nResearch question:\n{SCENARIO['question']}")
print(f"\nCorpus:\n{SCENARIO['corpus_note']}")
print(
    "\nSelect your analytical methods in Steps 4-8. Every choice must be "
    "justified in your report with explicit reference to your research "
    "question and the methodological literature."
)
print("\nDo not change your allocated scenario. Record it in your report.")

In [ ]:
#@title Assignment workflow

def _draw_workflow_diagram():
    """Render the assignment step workflow as a horizontal pipeline."""
    import matplotlib.patches as mpatches
    import matplotlib.pyplot as plt

    stages = [
        ("3",     "Load\nCorpus"),
        ("4\u20135",  "Pre-\nprocessing"),
        ("6\u20136c", "Sentiment\nAnalysis"),
        ("7\u20139",  "Secondary\nMetric"),
        ("10",    "Visuali-\nsation"),
        ("11\u201312","Manual\nValidation"),
        ("13\u201314","Submit &\nSummary"),
    ]
    colours = ["#4f46e5", "#0ea5e9", "#10b981", "#f59e0b", "#ef4444", "#8b5cf6", "#06b6d4"]

    fig, ax = plt.subplots(figsize=(13, 2.4))
    ax.set_xlim(0, len(stages))
    ax.set_ylim(0, 1)
    ax.axis("off")
    bw, bh, by = 0.82, 0.56, 0.22

    for i, ((code, label), col) in enumerate(zip(stages, colours)):
        cx = i + 0.5
        rect = mpatches.FancyBboxPatch(
            (cx - bw / 2, by), bw, bh,
            boxstyle="round,pad=0.04", linewidth=1.2,
            edgecolor="white", facecolor=col, zorder=3)
        ax.add_patch(rect)
        ax.text(cx, by + bh / 2 + 0.08, f"Step {code}",
                ha="center", va="center",
                fontsize=7.5, fontweight="bold", color="white", zorder=4)
        ax.text(cx, by + bh / 2 - 0.10, label,
                ha="center", va="center",
                fontsize=6.5, color="white", zorder=4, linespacing=1.3)
        if i < len(stages) - 1:
            ax.annotate("", xy=(i + 1 + 0.5 - bw / 2 - 0.02, by + bh / 2),
                        xytext=(cx + bw / 2 + 0.02, by + bh / 2),
                        arrowprops=dict(arrowstyle="->", color="#64748b", lw=1.4),
                        zorder=2)

    ax.set_title("Assignment workflow \u2014 complete steps in order",
                 fontsize=9, color="#475569", pad=6)
    plt.tight_layout()
    plt.show()

_draw_workflow_diagram()

In [ ]:
#@title Step 3 -- Load the corpus

# The corpus contains two sections for each firm and year:
#
#   item_1a  --  Risk Factors
#   item_7   --  Management's Discussion and Analysis (MD&A)
#
# You must decide which section(s) to analyse and justify that choice in
# your report. Three approaches are possible:
#
#   Single section:
#     df = corpus[corpus["section"] == "item_1a"]
#     df = corpus[corpus["section"] == "item_7"]
#
#   Both sections combined (treating the full narrative disclosure as the
#   unit of analysis):
#     df = corpus.copy()                          # use as-is, or:
#     df = corpus.groupby(["firm","category","year"], as_index=False).agg(
#              {"text": " ".join, "word_count": "sum"})
#
# The section choice has direct consequences for which linguistic phenomena
# are captured and must be defended with reference to the literature and
# your scenario's research question.

CORPUS_PATH = os.path.join(
    CORPUS_BASE_PATH,
    f"scenario_{ASSIGNED_SCENARIO_KEY.lower()}",
    "corpus.csv"
)

try:
    corpus = pd.read_csv(CORPUS_PATH)
    print(f"Corpus loaded from: {CORPUS_PATH}")
    print(f"Shape: {corpus.shape[0]} rows x {corpus.shape[1]} columns")
    print(f"Columns: {list(corpus.columns)}")
    print(f"\nUnique firms:    {corpus['firm'].nunique()}")
    print(f"Unique years:    {sorted(corpus['year'].unique())}")
    print(f"Sections:        {sorted(corpus['section'].unique())}")
    print(f"\nFirst 3 rows (transposed):")
    display(corpus.head(3).T)
except FileNotFoundError:
    print(f"ERROR: corpus not found at {CORPUS_PATH}.")
    print("Run `git pull origin main` to ensure the data files are present.")

## Pre-processing

You must implement a cleaning pipeline that prepares the raw 10-K text for downstream analysis and justify every decision in your report with explicit reference to Grimmer, Roberts and Stewart (2022). Four decisions are required, each with direct consequences for the validity of your results. Choices made without theoretical justification will be penalised in the marking scheme.

| Decision | What the choice affects |
|---|---|
| Stop-word list | Which words are excluded from the token set prior to analysis |
| Stemming vs. lemmatisation | How inflected word forms are handled; affects vocabulary size and the mapping of related terms |
| Number handling | Whether numeric tokens are retained in, removed from, or replaced within the token set |
| TF-IDF weighting | Whether term frequencies are adjusted by their rarity across the corpus before scoring |

In [ ]:
#@title Step 4 -- Record your pre-processing decisions

!pip install ipywidgets -q

import ipywidgets as widgets
from IPython.display import display

w_stopwords = widgets.Dropdown(
    options=["Standard NLTK stopwords",
             "Finance-adjusted (modal verbs and negation retained)"],
    description="Stop-word list:",
    style={"description_width": "200px"},
    layout=widgets.Layout(width="500px")
)
w_normalisation = widgets.Dropdown(
    options=["Lemmatisation (WordNetLemmatizer)", "Stemming (PorterStemmer)", "None"],
    description="Normalisation:",
    style={"description_width": "200px"},
    layout=widgets.Layout(width="500px")
)
w_numbers = widgets.Dropdown(
    options=["Remove all numeric tokens", "Retain as-is",
             "Replace with placeholder <NUM>"],
    description="Number handling:",
    style={"description_width": "200px"},
    layout=widgets.Layout(width="500px")
)
w_tfidf = widgets.Dropdown(
    options=["Yes -- apply TF-IDF weighting", "No -- use raw term counts"],
    description="TF-IDF weighting:",
    style={"description_width": "200px"},
    layout=widgets.Layout(width="500px")
)

# Live confirmation panel -- updates immediately when any dropdown changes
_out = widgets.Output()

def _show_selections(_change=None):
    _out.clear_output(wait=True)
    with _out:
        print("Choices recorded:")
        print(f"  Stop-word list:   {w_stopwords.value}")
        print(f"  Normalisation:    {w_normalisation.value}")
        print(f"  Number handling:  {w_numbers.value}")
        print(f"  TF-IDF weighting: {w_tfidf.value}")
        print()
        print("These values are stored in the Python session. They take effect")
        print("once you implement preprocess() in Step 5, and are submitted to")
        print("the module log when you run Step 13.")

for _w in [w_stopwords, w_normalisation, w_numbers, w_tfidf]:
    _w.observe(_show_selections, names="value")

display(w_stopwords, w_normalisation, w_numbers, w_tfidf, _out)
_show_selections()  # show initial state immediately

In [ ]:
#@title Step 5 -- Implement your pre-processing pipeline

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer


def preprocess(text: str) -> list:
    """
    Clean and tokenise a single document string.
    Wire your widget choices into each decision point below.

    Widget values available:
      w_stopwords.value     -- controls stop-word list
      w_normalisation.value -- controls stemming / lemmatisation
      w_numbers.value       -- controls number handling
      w_tfidf.value         -- TF-IDF is applied at corpus level; see Step 8
    """
    # YOUR CODE HERE
    pass


# Test on the first row of the corpus
try:
    sample_text = corpus["text"].iloc[0] if "text" in corpus.columns else ""
    tokens = preprocess(sample_text)
    if tokens is None:
        print("preprocess() has not been implemented yet -- add your code above, then re-run this cell.")
    else:
        print(f"First 30 tokens: {tokens[:30]}")
except NameError:
    print("corpus is not yet defined -- run Step 3 first, then re-run this cell.")

## Sentiment Analysis

You must apply one sentiment engine to the corpus and justify your choice in the report on at least three criteria: transparency of the underlying word list or model weights; domain specificity of the method to the text being analysed; and computational requirements relative to the corpus size. Four approaches are available.

The **Loughran-McDonald (LM) dictionary** and **Harvard General Inquirer (Harvard IV)** are word-list approaches in which sentiment is measured by matching pre-processed tokens against fixed positive and negative word sets. Both dictionaries were constructed from different source corpora and reflect different theoretical assumptions about what constitutes sentiment in text. When you select `lm_dictionary`, an uncertainty word count is computed automatically alongside the sentiment scores.

The **Naive Bayes** and **Logistic Regression** classifiers are supervised methods trained on the pre-labelled sentences provided in `nb_seed_labels.csv`. Your report must evaluate classifier performance using precision, recall, and F1, and must discuss the relationship between training set characteristics and the validity of predictions applied to the full corpus.

In [ ]:
#@title Step 6 -- Select your sentiment model

w_sentiment = widgets.Dropdown(
    options=["lm_dictionary", "harvard_iv", "naive_bayes", "logistic_regression"],
    description="Sentiment model:",
    style={"description_width": "200px"},
    layout=widgets.Layout(width="500px")
)
print("Select your sentiment model, then run the appropriate implementation cell below.")
print("  Dictionary approaches (LM or Harvard IV) → run Step 6a")
print("  ML classifier approaches (Naive Bayes or Logistic Regression) → run Step 6b")
display(w_sentiment)

In [ ]:
#@title Step 6a -- Dictionary sentiment (LM or Harvard IV)

# Run this cell if w_sentiment.value is "lm_dictionary" or "harvard_iv".
# Skip it if you chose a classifier approach; run Step 6b instead.

if w_sentiment.value in ("lm_dictionary", "harvard_iv"):

    dict_path = LM_DICT_PATH if w_sentiment.value == "lm_dictionary" else HARVARD_DICT_PATH

    try:
        d = pd.read_csv(dict_path)
        positive_words    = set(d[d["Positive"]    != 0]["Word"].str.lower())
        negative_words    = set(d[d["Negative"]    != 0]["Word"].str.lower())
        uncertainty_words = (set(d[d["Uncertainty"] != 0]["Word"].str.lower())
                             if "Uncertainty" in d.columns else set())
        unc_note = f", {len(uncertainty_words)} uncertainty" if uncertainty_words else ""
        print(f"{w_sentiment.value} loaded: "
              f"{len(positive_words)} positive, {len(negative_words)} negative{unc_note}.")
    except FileNotFoundError:
        print(f"ERROR: dictionary not found at {dict_path}.")
        print("Run `git pull origin main` to ensure data files are present.")
        positive_words, negative_words, uncertainty_words = set(), set(), set()


    def dict_sentiment(text, pos_words, neg_words, unc_words):
        """
        Compute dictionary sentiment scores for a single document.

        Tokenisation: uses preprocess() if it has been implemented in Step 5;
        falls back to whitespace splitting if preprocess() returns None or
        has not yet been written.

        Returns a dict with:
          pos_count       -- positive word matches
          neg_count       -- negative word matches
          net_sentiment   -- pos_count - neg_count
          sentiment_score -- (pos - neg) / (pos + neg), or np.nan if denominator == 0
          unc_count       -- uncertainty word matches (LM only; 0 for Harvard IV)
        """
        try:
            _tokens = preprocess(str(text))
            tokens = [t.lower() for t in _tokens] if _tokens else \
                     [t.lower() for t in str(text).split()]
        except Exception:
            tokens = [t.lower() for t in str(text).split()]

        pos   = sum(1 for t in tokens if t in pos_words)
        neg   = sum(1 for t in tokens if t in neg_words)
        unc   = sum(1 for t in tokens if t in unc_words)
        denom = pos + neg
        score = (pos - neg) / denom if denom > 0 else np.nan
        return {"pos_count": pos, "neg_count": neg,
                "net_sentiment": pos - neg, "sentiment_score": score,
                "unc_count": unc}


    if "text" in corpus.columns and positive_words:
        results = corpus["text"].apply(
            lambda t: dict_sentiment(t, positive_words, negative_words, uncertainty_words)
        )
        corpus["dict_pos"]   = results.apply(lambda r: r["pos_count"])
        corpus["dict_neg"]   = results.apply(lambda r: r["neg_count"])
        corpus["dict_net"]   = results.apply(lambda r: r["net_sentiment"])
        corpus["dict_score"] = results.apply(lambda r: r["sentiment_score"])
        if uncertainty_words:
            corpus["dict_unc"] = results.apply(lambda r: r["unc_count"])
        print("Dictionary sentiment scores computed.")
        display(corpus[["dict_pos", "dict_neg", "dict_score"]].describe())

else:
    print(f"Skipping Step 6a: sentiment model is '{w_sentiment.value}'. Run Step 6b instead.")

In [ ]:
#@title Step 6b -- ML classifier sentiment (Naive Bayes or Logistic Regression)

# Run this cell if w_sentiment.value is "naive_bayes" or "logistic_regression".
# Skip it if you chose a dictionary approach; run Step 6a instead.

from sklearn.linear_model import LogisticRegression

if w_sentiment.value in ("naive_bayes", "logistic_regression"):

    # Load seed labels (pre-labelled sentences used as training data)
    try:
        seed_labels = pd.read_csv(SEED_LABELS_PATH)
        print(f"Seed labels loaded: {seed_labels.shape[0]} rows")
        display(seed_labels["label"].value_counts().to_frame())
    except FileNotFoundError:
        print(f"ERROR: seed labels not found at {SEED_LABELS_PATH}.")
        print("Run `git pull origin main` to ensure data files are present.")
        seed_labels = None

    if seed_labels is not None:
        # YOUR CODE HERE
        #
        # seed_labels has two columns: "sentence" (text) and "label"
        # (Positive / Negative / Neutral).
        # Train a classifier on these labels, evaluate it on a held-out
        # test split, then apply it to score every document in the corpus.
        #
        # Suggested skeleton:
        #
        # vectorizer = TfidfVectorizer(max_features=5000)
        # X = vectorizer.fit_transform(seed_labels["sentence"])
        # y = seed_labels["label"]
        # X_train, X_test, y_train, y_test = train_test_split(
        #     X, y, test_size=0.2, random_state=42
        # )
        #
        # if w_sentiment.value == "naive_bayes":
        #     clf = MultinomialNB()
        # else:  # logistic_regression
        #     clf = LogisticRegression(max_iter=1000, random_state=42)
        #
        # clf.fit(X_train, y_train)
        # y_pred = clf.predict(X_test)
        # print(classification_report(y_test, y_pred))
        # print(confusion_matrix(y_test, y_pred))
        #
        # # Apply classifier to the full corpus
        # corpus_vectors = vectorizer.transform(corpus["text"].fillna(""))
        # corpus["ml_sentiment"] = clf.predict(corpus_vectors)
        # display(corpus["ml_sentiment"].value_counts().to_frame())
        pass

else:
    print(f"Skipping Step 6b: sentiment model is '{w_sentiment.value}'. Run Step 6a instead.")

In [ ]:
#@title Step 6c -- Sentiment score table

# This cell pivots your sentiment scores into a company x year matrix.
# Run it after Step 6a or Step 6b has completed successfully.
#
# By default the full corpus is used (all sections).
# If you have filtered to a specific section, replace `_df` below
# with your filtered DataFrame before running.

_df = corpus   # replace with e.g. corpus[corpus["section"] == "item_1a"] if needed

score_col = ("dict_score"   if "dict_score"   in _df.columns else
             "ml_sentiment" if "ml_sentiment" in _df.columns else None)

if score_col is None:
    print("No sentiment scores found in the corpus. Run Step 6a or Step 6b first.")
else:
    if pd.api.types.is_numeric_dtype(_df[score_col]):
        pivot = (_df
                 .pivot_table(index=["firm", "category"],
                              columns="year",
                              values=score_col,
                              aggfunc="mean")
                 .round(4))
    else:
        pivot = (_df
                 .pivot_table(index=["firm", "category"],
                              columns="year",
                              values=score_col,
                              aggfunc=lambda x: x.mode().iloc[0] if not x.empty else ""))

    pivot = pivot.reset_index()
    pivot.columns.name = None
    year_cols = sorted([c for c in pivot.columns if c not in ("firm", "category")])
    pivot = pivot[["firm", "category"] + year_cols]
    pivot.rename(columns={"firm": "Firm", "category": "Industry"}, inplace=True)

    print(f"Sentiment scores ({w_sentiment.value}) \u2014 companies as rows, years as columns")
    print("Copy this table into your reflective document.\n")
    display(pivot)

    # LM uncertainty scores (computed automatically when lm_dictionary is selected)
    if "dict_unc" in _df.columns:
        unc_pivot = (_df
                     .pivot_table(index=["firm", "category"],
                                  columns="year",
                                  values="dict_unc",
                                  aggfunc="mean")
                     .round(1)
                     .reset_index())
        unc_pivot.columns.name = None
        year_cols_u = sorted([c for c in unc_pivot.columns if c not in ("firm", "category")])
        unc_pivot = unc_pivot[["firm", "category"] + year_cols_u]
        unc_pivot.rename(columns={"firm": "Firm", "category": "Industry"}, inplace=True)
        print("\nUncertainty word counts (LM) \u2014 copy this table into your reflective document.\n")
        display(unc_pivot)

## Secondary Metric

You must supplement your sentiment analysis with one secondary metric and explain in your report what incremental information it provides that sentiment alone cannot. Three options are scaffolded: the Gunning Fog readability index (Li, 2008), which captures disclosure complexity and has been linked to earnings persistence; year-on-year cosine similarity (Brown and Tucker, 2011), which distinguishes substantive revision from boilerplate repetition; and Latent Dirichlet Allocation (Grimmer et al., 2022, Ch. 13), which identifies latent thematic structure without requiring a predefined vocabulary. Your choice should be theoretically motivated by your scenario's research question.

In [ ]:
#@title Step 7 -- Select your secondary metric

w_secondary = widgets.Dropdown(
    options=["fog_index", "cosine_similarity", "lda"],
    description="Secondary metric:",
    style={"description_width": "200px"},
    layout=widgets.Layout(width="500px")
)
print("Select your secondary metric, then implement it in the scaffold below.")
display(w_secondary)

In [ ]:
#@title Step 8 -- Implement your secondary metric

# LDA requires gensim. Install it now if that is your chosen metric.
if w_secondary.value == "lda":
    import subprocess
    subprocess.run(["pip", "install", "gensim", "-q"], capture_output=True)
    print("gensim installed.")


def gunning_fog(text: str) -> float:
    """
    Gunning Fog Index.
    Formula: 0.4 * (avg_sentence_length + pct_complex_words)
    Complex words: words with 3 or more syllables.
    Reference: Li (2008), Journal of Accounting and Economics.
    """
    # YOUR CODE HERE
    pass


def cosine_year_on_year(text_year_t: str, text_year_t1: str) -> float:
    """
    TF-IDF cosine similarity between two texts from consecutive years.
    Returns a float in [0, 1]; values near 1 indicate minimal revision.
    Reference: Brown and Tucker (2011), Journal of Accounting Research.
    """
    # YOUR CODE HERE
    pass


def run_lda(texts: list, n_topics: int = 10) -> tuple:
    """
    Fit an LDA model to pre-processed texts.
    Returns: (lda_model, topic_word_matrix, document_topic_matrix)
    Reference: Grimmer et al. (2022), Text as Data, Ch. 13.
    """
    # YOUR CODE HERE
    pass

In [ ]:
#@title Step 9 -- Secondary metric score table

# Run this cell after implementing your secondary metric in Step 8.
# It will generate a company x year table ready to copy into your document.
#
# For Gunning Fog: apply gunning_fog() to the corpus before running:
#   corpus["fog"] = corpus["text"].apply(gunning_fog)
#
# For cosine similarity: build a DataFrame named `similarity_df` with
#   columns: firm, category, year_pair (e.g. "2019-2020"), similarity
#   then uncomment the pivot block in the "cosine_similarity" branch below.
#
# For LDA: add a "dominant_topic" column after fitting your model:
#   corpus["dominant_topic"] = document_topic_matrix.argmax(axis=1)

_df = corpus   # replace with a filtered DataFrame if needed

if w_secondary.value == "fog_index":

    if "fog" not in _df.columns:
        print("No 'fog' column found.")
        print("Apply gunning_fog() to your DataFrame first:")
        print("  corpus['fog'] = corpus['text'].apply(gunning_fog)")
    else:
        fog_pivot = (_df
                     .pivot_table(index=["firm", "category"],
                                  columns="year",
                                  values="fog",
                                  aggfunc="mean")
                     .round(2)
                     .reset_index())
        fog_pivot.columns.name = None
        year_cols = sorted([c for c in fog_pivot.columns if c not in ("firm", "category")])
        fog_pivot = fog_pivot[["firm", "category"] + year_cols]
        fog_pivot.rename(columns={"firm": "Firm", "category": "Industry"}, inplace=True)
        print("Gunning Fog scores \u2014 companies as rows, years as columns")
        print("Copy this table into your reflective document.\n")
        display(fog_pivot)

elif w_secondary.value == "cosine_similarity":

    # After computing pairwise similarities, store them in a DataFrame named
    # `similarity_df` with columns: firm, category, year_pair, similarity
    # then uncomment and run the block below.
    #
    # sim_pivot = (similarity_df
    #              .pivot_table(index=["firm", "category"],
    #                           columns="year_pair",
    #                           values="similarity",
    #                           aggfunc="mean")
    #              .round(4)
    #              .reset_index())
    # sim_pivot.columns.name = None
    # sim_pivot.rename(columns={"firm": "Firm", "category": "Industry"}, inplace=True)
    # print("Year-on-year cosine similarity \u2014 companies as rows, year pairs as columns")
    # print("Copy this table into your reflective document.\n")
    # display(sim_pivot)
    print("Cosine similarity: uncomment the block above after computing similarity_df.")

elif w_secondary.value == "lda":

    if "dominant_topic" not in _df.columns:
        print("No 'dominant_topic' column found.")
        print("Add it after fitting your LDA model:")
        print("  corpus['dominant_topic'] = document_topic_matrix.argmax(axis=1)")
    else:
        lda_pivot = (_df
                     .pivot_table(index=["firm", "category"],
                                  columns="year",
                                  values="dominant_topic",
                                  aggfunc=lambda x: int(x.mode().iloc[0]) if not x.empty else "")
                     .reset_index())
        lda_pivot.columns.name = None
        year_cols = sorted([c for c in lda_pivot.columns if c not in ("firm", "category")])
        lda_pivot = lda_pivot[["firm", "category"] + year_cols]
        lda_pivot.rename(columns={"firm": "Firm", "category": "Industry"}, inplace=True)
        print("Dominant LDA topic \u2014 companies as rows, years as columns")
        print("Copy this table into your reflective document.\n")
        display(lda_pivot)

## Visualisation

You must produce a minimum of two professional-standard visualisations and include them in your report. Each figure must have clearly labelled axes, a descriptive title, and a legend where multiple series are shown. Save each figure to disk before calling `plt.show()` using `plt.savefig('figN.png', dpi=150, bbox_inches='tight')` so that it persists across sessions and can be incorporated directly into your written submission.

In [ ]:
#@title Step 10 -- Produce visualisations

# YOUR CODE HERE
#
# Reminder: call plt.savefig() before plt.show() so the file is written
# to disk before the figure is cleared from memory.
# Example:
#   plt.savefig("fig1.png", dpi=150, bbox_inches="tight")
#   plt.show()
pass

## Manual Validation

Automated sentiment methods require empirical validation. You must draw a random sample of 100 sentences from the corpus, classify each sentence as positive, negative, or neutral by reading it in context, and compare your human judgements to the automated scores. Quantify the machine-human gap using a metric of your choice (e.g. percentage agreement, Cohen's kappa) and discuss where failures are concentrated: hedged language (*it is possible that...*), negation (*did not improve*), and context-dependent terms (*risk* as threat versus *risk management* as competence) are the most common failure modes in financial text and should receive explicit attention in your report.

In [ ]:
#@title Step 11 -- Draw random sample for manual validation

random.seed(42)

if "text" in corpus.columns:
    all_sentences = []
    for _, row in corpus.iterrows():
        for sent in str(row["text"]).split("."):
            sent = sent.strip()
            if len(sent.split()) >= 5:
                all_sentences.append({
                    "firm": row.get("firm", ""),
                    "year": row.get("year", ""),
                    "sentence_text": sent
                })

    sample = random.sample(all_sentences, min(100, len(all_sentences)))
    validation_df = pd.DataFrame(sample)
    validation_df.insert(0, "sentence_id", range(1, len(validation_df) + 1))
    validation_df["human_label"]     = ""
    validation_df["automated_label"] = ""

    validation_df.to_csv("manual_validation_sample.csv", index=False)
    print("manual_validation_sample.csv written.")
    print(f"Rows: {len(validation_df)}")
    display(validation_df.head())
else:
    print("No 'text' column found in corpus. Load the corpus first (Step 3).")

In [ ]:
#@title Step 12 -- Review seed labels

# The seed labels provide the training data for the Naive Bayes and
# Logistic Regression approaches in Step 6b.
#
# If you chose a dictionary approach in Step 6a, you can skip this cell.
#
# If you are implementing a Naive Bayes or Logistic Regression classifier,
# run this cell BEFORE implementing Step 6b. Review the label distribution
# and read a sample of sentences to satisfy yourself that the label
# definitions are consistent and that the sentences are representative of
# the 10-K language in your corpus. Training data quality directly affects
# the validity of predictions applied to the full corpus.

try:
    seed_labels = pd.read_csv(SEED_LABELS_PATH)
    print(f"Seed labels loaded: {seed_labels.shape[0]} rows x {seed_labels.shape[1]} columns")
    print(f"Columns: {list(seed_labels.columns)}")
    print(f"\nLabel distribution:")
    display(seed_labels["label"].value_counts().to_frame())
    print(f"\nSample sentences:")
    display(seed_labels.sample(5, random_state=1).reset_index(drop=True))
except FileNotFoundError:
    print(f"WARNING: seed labels file not found at {SEED_LABELS_PATH}.")
    print("Run `git pull origin main` to ensure the file is present.")

In [ ]:
#@title Step 13 -- Submit your methodological choices

# Run this cell after finalising all choices above.
# You may re-run it if you change a choice; each submission is
# recorded separately and the submission_count column will increment.


def submit_choices(apps_script_url, student_id, scenario_id):
    if not apps_script_url:
        print("APPS_SCRIPT_URL is not set. Choices not submitted.")
        print("Contact your module leader if you are unsure how to proceed.")
        return

    payload = {
        "student_id":           student_id,
        "scenario":             scenario_id,
        "stopword_list":        w_stopwords.value,
        "normalisation_method": w_normalisation.value,
        "number_handling":      w_numbers.value,
        "tfidf_weighting":      w_tfidf.value,
        "sentiment_model":      w_sentiment.value,
        "secondary_metric":     w_secondary.value,
    }

    try:
        response = requests.post(
            apps_script_url,
            data=json.dumps(payload),
            headers={"Content-Type": "application/json"},
            timeout=15
        )
        result = response.json()
        if result.get("status") == "success":
            print(f"Choices submitted. This is submission number "
                  f"{result.get('submission', '?')} "
                  f"for student ID {student_id}.")
        else:
            print(f"Submission returned an error: "
                  f"{result.get('message', 'unknown error')}")
    except requests.exceptions.Timeout:
        print("The request timed out. Check your internet connection and try again.")
    except Exception as ex:
        print(f"Submission failed: {ex}")


submit_choices(APPS_SCRIPT_URL, STUDENT_ID, ASSIGNED_SCENARIO_KEY)

In [ ]:
#@title Step 14 -- Your choices summary

# This cell generates a formatted table of your methodological choices.
# Copy it into your reflective document.

summary_data = {
    "Choice": [
        "Student ID",
        "Scenario",
        "Stop-word list",
        "Normalisation",
        "Number handling",
        "TF-IDF weighting",
        "Sentiment model",
        "Secondary metric",
    ],
    "Selection": [
        STUDENT_ID,
        f"{SCENARIO['id']} \u2014 {SCENARIO['title']}",
        w_stopwords.value,
        w_normalisation.value,
        w_numbers.value,
        w_tfidf.value,
        w_sentiment.value,
        w_secondary.value,
    ]
}

summary_df = pd.DataFrame(summary_data).set_index("Choice")
print("Methodological choices \u2014 copy this table into your reflective document:\n")
display(summary_df)